# LeetCode 2885 — Rename Columns bằng Python (pandas)

Link đề: https://leetcode.com/problems/rename-columns/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def renameColumns(students: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `students` có các cột `id`, `first`, `last`, `age`; **đổi tên cột** thành `student_id`, `first_name`, `last_name`, `age_in_years`, dữ liệu không đổi.
2. Hiểu **tên cột là `df.columns`** (một `Index`), và `dict` ánh xạ **tên cũ → tên mới**.
3. Đi từ cách **tự duyệt tên cột và tra từ điển** → **`rename(columns=mapping)`** → **gán trực tiếp `df.columns = [...]`** → **`set_axis`** → **`rename(columns=hàm)`**.
4. Phân biệt đổi tên **theo tên** (an toàn) với đổi tên **theo vị trí** (dễ sai khi thứ tự cột thay đổi).
5. Tránh các lỗi kinh điển: quên `columns=` (đổi nhãn hàng thay vì cột), quên gán lại, `inplace=True` rồi `return None`, gõ sai tên cũ (bị lờ đi âm thầm), danh sách tên mới sai độ dài.

## 2. Đề bài & ví dụ

Có một bảng `students` với các cột `id`, `first`, `last`, `age`.

Hãy viết hàm **đổi tên các cột** như sau và trả về bảng mới:

| tên cũ | tên mới |
|---|---|
| `id` | `student_id` |
| `first` | `first_name` |
| `last` | `last_name` |
| `age` | `age_in_years` |

**Ví dụ**

- Input: DataFrame `students`

| id | first | last | age |
|---|---|---|---|
| 1 | Mason | King | 6 |
| 2 | Ava | Wright | 7 |
| 3 | Taylor | Hall | 16 |
| 4 | Georgia | Thompson | 18 |
| 5 | Thomas | Moore | 10 |

- Output:

| student_id | first_name | last_name | age_in_years |
|---|---|---|---|
| 1 | Mason | King | 6 |
| 2 | Ava | Wright | 7 |
| 3 | Taylor | Hall | 16 |
| 4 | Georgia | Thompson | 18 |
| 5 | Thomas | Moore | 10 |

- Giải thích: chỉ **tiêu đề cột** thay đổi; mọi giá trị, thứ tự hàng và thứ tự cột giữ nguyên.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "first": ["Mason", "Ava", "Taylor", "Georgia", "Thomas"],
    "last": ["King", "Wright", "Hall", "Thompson", "Moore"],
    "age": [6, 7, 16, 18, 10],
})
print(students)
print()
print(students.columns.tolist())
print("pandas", pd.__version__)

   id    first      last  age
0   1    Mason      King    6
1   2      Ava    Wright    7
2   3   Taylor      Hall   16
3   4  Georgia  Thompson   18
4   5   Thomas     Moore   10

['id', 'first', 'last', 'age']
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. `df.columns` — danh sách tên cột

`df.columns` là một đối tượng `Index` chứa **tên các cột theo thứ tự**. Có thể đổi sang `list` bằng `.tolist()`, lấy độ dài bằng `len`, duyệt bằng `for`. Tên cột cũng là "chìa khóa" để lấy cột: `df["age"]`.

In [2]:
cols = students.columns
print(type(cols).__name__)
print(cols.tolist())           # ['id', 'first', 'last', 'age']
print(len(cols))               # 4
for position, name in enumerate(cols):
    print(position, name)

Index
['id', 'first', 'last', 'age']
4
0 id
1 first
2 last
3 age


### 3.2. `dict` ánh xạ tên cũ → tên mới

`dict` lưu cặp **khóa → giá trị**. Ta dùng khóa là **tên cũ**, giá trị là **tên mới**. `mapping.get(key, mặc_định)` tra khóa; nếu khóa không có thì trả `mặc_định` (ở đây ta cho mặc định là chính tên cũ, nghĩa là "cột không nằm trong bảng đổi thì giữ tên").

In [3]:
mapping = {
    "id": "student_id",
    "first": "first_name",
    "last": "last_name",
    "age": "age_in_years",
}
print(mapping["id"])                     # student_id
print(mapping.get("age", "age"))         # age_in_years
print(mapping.get("grade", "grade"))     # grade: không có trong mapping -> giữ nguyên
print("first" in mapping)                # True

student_id
age_in_years
grade
True


### 3.3. `DataFrame.rename(columns=...)`

`df.rename(columns=mapping)` trả về **bảng mới** có tên cột đã đổi **theo từ điển**. Cột nào không có trong `mapping` thì giữ tên cũ. Đổi tên **theo tên** nên **không phụ thuộc thứ tự cột**.

In [4]:
renamed = students.rename(columns=mapping)
print(renamed.columns.tolist())
print("bảng gốc vẫn:", students.columns.tolist())      # rename không sửa bảng cũ

['student_id', 'first_name', 'last_name', 'age_in_years']
bảng gốc vẫn: ['id', 'first', 'last', 'age']


### 3.4. Gán trực tiếp `df.columns = [...]`

Có thể gán thẳng **danh sách tên mới** cho `df.columns`. Danh sách phải **cùng độ dài** với số cột và được ghép **theo vị trí**: phần tử thứ 0 thành tên cột 0, v.v. Cách này **sửa tại chỗ** bảng `df`. Rất gọn nhưng dễ sai nếu thứ tự cột không như bạn tưởng.

In [5]:
tmp = students.copy()                     # làm trên bản sao để giữ dữ liệu mẫu
tmp.columns = ["student_id", "first_name", "last_name", "age_in_years"]
print(tmp.columns.tolist())
print(students.columns.tolist())           # bảng gốc vẫn nguyên vì ta sửa bản sao

['student_id', 'first_name', 'last_name', 'age_in_years']
['id', 'first', 'last', 'age']


### 3.5. `set_axis` — gán nhãn và trả về bảng mới

`df.set_axis(danh_sách, axis=1)` giống gán `df.columns = danh_sách` nhưng **trả về bảng mới** và không sửa `df`, nên nối chuỗi được (`df.set_axis(...).sort_index()`). `axis=1` nghĩa là "trục cột"; `axis=0` là trục hàng.

In [6]:
new_df = students.set_axis(["student_id", "first_name", "last_name", "age_in_years"], axis=1)
print(new_df.columns.tolist())
print(students.columns.tolist())           # không bị đổi

['student_id', 'first_name', 'last_name', 'age_in_years']
['id', 'first', 'last', 'age']


### 3.6. `rename(columns=hàm)`

Thay vì từ điển, truyền **một hàm** nhận tên cũ và trả tên mới; hữu ích khi đổi hàng loạt theo quy tắc (viết hoa, thêm tiền tố, thay dấu cách...). Ví dụ: `str.upper` biến `id` thành `ID`.

In [7]:
print(students.rename(columns=str.upper).columns.tolist())             # ['ID', 'FIRST', 'LAST', 'AGE']
print(students.rename(columns=lambda c: "col_" + c).columns.tolist())  # thêm tiền tố

['ID', 'FIRST', 'LAST', 'AGE']
['col_id', 'col_first', 'col_last', 'col_age']


### 3.7. Quên `columns=` thì đổi nhãn HÀNG, không phải cột

`df.rename(mapping)` (không có `columns=`) áp dụng cho **index (nhãn hàng)** theo mặc định. Vì nhãn hàng là `0, 1, 2...` không khớp khóa `"id"`, nên **không có gì thay đổi**, và pandas cũng **không báo lỗi**. Đây là lỗi âm thầm rất hay gặp.

In [8]:
wrong = students.rename(mapping)            # thiếu columns=
print(wrong.columns.tolist())                # vẫn tên cũ!
print(wrong.equals(students))                # True: bảng y hệt

['id', 'first', 'last', 'age']
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một bảng tính có **hàng tiêu đề**. Việc ta làm là **thay chữ trên hàng tiêu đề**, không động tới các ô dữ liệu bên dưới.

| vị trí cột | tên cũ | tra bảng đổi | tên mới |
|---|---|---|---|
| 0 | `id` | `id` -> `student_id` | `student_id` |
| 1 | `first` | `first` -> `first_name` | `first_name` |
| 2 | `last` | `last` -> `last_name` | `last_name` |
| 3 | `age` | `age` -> `age_in_years` | `age_in_years` |

Thuật toán: **với mỗi tên cột cũ, tra tên mới trong từ điển (không có thì giữ nguyên), rồi đặt lại tiêu đề**. Mọi cách ở dưới chỉ là những cách khác nhau để thực hiện đúng việc này.

## 5. Cách 1 — Brute Force / baseline: tự duyệt tên cột và tra từ điển

**Ý tưởng.** Duyệt từng tên cột cũ, tra `mapping.get(name, name)`, gom vào danh sách `new_names`; cuối cùng gán `df.columns = new_names`. Đây là việc mà `rename` làm hộ ta.

In [9]:
def rename_loop(students: pd.DataFrame) -> pd.DataFrame:
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    df = students.copy()                       # không đụng bảng gốc
    new_names = []
    for old in df.columns:                     # duyệt từng tên cột cũ
        new_names.append(mapping.get(old, old))   # tra tên mới, không có thì giữ tên cũ
    df.columns = new_names                     # đặt lại tiêu đề
    return df


print(rename_loop(students))

   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7
2           3     Taylor      Hall            16
3           4    Georgia  Thompson            18
4           5     Thomas     Moore            10


In [10]:
res = rename_loop(students)
print("tên mới:", res.columns.tolist())
print("dữ liệu không đổi:", (res.to_numpy() == students.to_numpy()).all())

tên mới: ['student_id', 'first_name', 'last_name', 'age_in_years']
dữ liệu không đổi: True


**Độ phức tạp (Cách 1).** Gọi `c` là số cột. Vòng lặp `c` lần, mỗi lần tra từ điển `O(1)` trung bình: **Time `O(c)`** cho phần đổi tên; danh sách mới `c` phần tử: **Space `O(c)`** (cộng bản sao bảng nếu có `copy`).

**Nút thắt.** Phải tự viết `mapping`, `copy`, `get`... pandas có sẵn `rename` làm gọn tất cả.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`students.rename(columns={"id": "student_id", ...})`

- `columns=`: nói rõ là đổi **tên cột** (không phải nhãn hàng);
- cột không có trong từ điển giữ nguyên tên;
- dữ liệu, thứ tự hàng, thứ tự cột không đổi;
- trả về **bảng mới**.

Các cách khác: **gán `df.columns`** theo vị trí (Cách 3), **`set_axis`** (Cách 4), **`rename(columns=hàm)`** (Cách 5). Chọn cách nào tùy tình huống: đổi vài cột thì `rename` theo tên; đổi **toàn bộ** cột theo đúng thứ tự thì gán danh sách cũng ổn.

## 7. Các cách tiếp theo

### Cách 2 — `rename(columns=mapping)` (cách chuẩn)

**Giải thích.** Một lời gọi, đổi theo tên, không sửa bảng đầu vào (mặc định `inplace=False`).

In [11]:
def rename_rename(students: pd.DataFrame) -> pd.DataFrame:
    return students.rename(columns={
        "id": "student_id",
        "first": "first_name",
        "last": "last_name",
        "age": "age_in_years",
    })


print(rename_rename(students))

   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7
2           3     Taylor      Hall            16
3           4    Georgia  Thompson            18
4           5     Thomas     Moore            10


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def renameColumns(students: pd.DataFrame) -> pd.DataFrame:
    return students.rename(columns={
        "id": "student_id",
        "first": "first_name",
        "last": "last_name",
        "age": "age_in_years",
    })

**Độ phức tạp Cách 2:** **Time `O(c)`** để đổi tên (dữ liệu không bị xử lý; pandas có thể dùng chung bộ nhớ dữ liệu nhờ copy-on-write, còn nếu sao chép thì `O(m·c)`), **Space `O(c)` đến `O(m·c)`** tùy việc sao chép.

### Cách 3 — Gán `df.columns = [...]` theo vị trí

**Giải thích.** Gán danh sách tên mới **theo đúng thứ tự cột hiện tại**. Ngắn gọn nhưng phụ thuộc thứ tự: nếu thứ tự cột của bảng đầu vào khác dự kiến, tên sẽ bị gán nhầm cột mà không có lỗi nào báo.

In [13]:
def rename_assign(students: pd.DataFrame) -> pd.DataFrame:
    df = students.copy()
    df.columns = ["student_id", "first_name", "last_name", "age_in_years"]
    return df


print(rename_assign(students))

   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7
2           3     Taylor      Hall            16
3           4    Georgia  Thompson            18
4           5     Thomas     Moore            10


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def renameColumns(students: pd.DataFrame) -> pd.DataFrame:
    students.columns = ["student_id", "first_name", "last_name", "age_in_years"]
    return students

**Độ phức tạp Cách 3:** **Time `O(c)`**, **Space `O(c)`** (danh sách tên mới).

### Cách 4 — `set_axis(..., axis=1)`

**Giải thích.** Giống Cách 3 nhưng **trả bảng mới** (không sửa đầu vào), hợp với method chaining.

In [15]:
def rename_set_axis(students: pd.DataFrame) -> pd.DataFrame:
    return students.set_axis(["student_id", "first_name", "last_name", "age_in_years"], axis=1)


before = students.copy()
print(rename_set_axis(students))
pd.testing.assert_frame_equal(students, before)
print("đầu vào không đổi")

   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7
2           3     Taylor      Hall            16
3           4    Georgia  Thompson            18
4           5     Thomas     Moore            10
đầu vào không đổi


**Độ phức tạp Cách 4:** **Time `O(c)`**, **Space `O(c)` đến `O(m·c)`**.

### Cách 5 — `rename(columns=hàm)`

**Giải thích.** Viết một hàm `to_new(name)` thực hiện tra từ điển rồi đưa vào `rename`. Hữu ích khi quy tắc đổi tên là một **công thức** thay vì một bảng tra cứu. Ở đây ta dùng `mapping.get` như Cách 1 nhưng để pandas lo phần còn lại.

In [16]:
def rename_func(students: pd.DataFrame) -> pd.DataFrame:
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    return students.rename(columns=lambda old: mapping.get(old, old))


print(rename_func(students).columns.tolist())

['student_id', 'first_name', 'last_name', 'age_in_years']


**Độ phức tạp Cách 5:** **Time `O(c)`** (gọi hàm một lần cho mỗi tên cột), **Space `O(c)`**.

> Cách 2 là cách nên dùng: đổi theo **tên**, đọc là hiểu, không phụ thuộc thứ tự cột.

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết, in từng tên cột; (b) pipeline `rename` với bảng đổi tên.

In [17]:
def trace_loop(students):
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    new_names = []
    for step, old in enumerate(students.columns, start=1):
        new = mapping.get(old, old)
        print(f"bước {step}: tên cũ={old!r} -> tra mapping -> {new!r}")
        new_names.append(new)
        print(f"          new_names = {new_names}")
    df = students.copy()
    df.columns = new_names
    print("\nđặt df.columns =", df.columns.tolist())
    return df


_ = trace_loop(students)

bước 1: tên cũ='id' -> tra mapping -> 'student_id'
          new_names = ['student_id']
bước 2: tên cũ='first' -> tra mapping -> 'first_name'
          new_names = ['student_id', 'first_name']
bước 3: tên cũ='last' -> tra mapping -> 'last_name'
          new_names = ['student_id', 'first_name', 'last_name']
bước 4: tên cũ='age' -> tra mapping -> 'age_in_years'
          new_names = ['student_id', 'first_name', 'last_name', 'age_in_years']

đặt df.columns = ['student_id', 'first_name', 'last_name', 'age_in_years']


In [18]:
def trace_rename(students):
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    print("BƯỚC 0 - bảng gốc", students.shape, students.columns.tolist())
    print(students.head(2))

    out = students.rename(columns=mapping)
    print("\nBƯỚC 1 - rename(columns=mapping):", out.columns.tolist())
    print(out.head(2))

    print("\nBƯỚC 2 - kiểm tra dữ liệu không đổi")
    print("hình dạng:", students.shape, "->", out.shape)
    print("giá trị bằng nhau:", (students.to_numpy() == out.to_numpy()).all())
    print("bảng gốc còn tên cũ:", students.columns.tolist())
    return out


_ = trace_rename(students)

BƯỚC 0 - bảng gốc (5, 4) ['id', 'first', 'last', 'age']
   id  first    last  age
0   1  Mason    King    6
1   2    Ava  Wright    7

BƯỚC 1 - rename(columns=mapping): ['student_id', 'first_name', 'last_name', 'age_in_years']
   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7

BƯỚC 2 - kiểm tra dữ liệu không đổi
hình dạng: (5, 4) -> (5, 4)
giá trị bằng nhau: True
bảng gốc còn tên cũ: ['id', 'first', 'last', 'age']


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | `students.columns` | `['id', 'first', 'last', 'age']` |
| 1 | tra `mapping` từng tên | `['student_id', 'first_name', 'last_name', 'age_in_years']` |
| 2 | bảng mới | cùng hình dạng `(5, 4)`, cùng dữ liệu, chỉ tiêu đề khác |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def renameColumns(students: pd.DataFrame) -> pd.DataFrame:
    return students.rename(columns={
        "id": "student_id",
        "first": "first_name",
        "last": "last_name",
        "age": "age_in_years",
    })


print(renameColumns(students))

   student_id first_name last_name  age_in_years
0           1      Mason      King             6
1           2        Ava    Wright             7
2           3     Taylor      Hall            16
3           4    Georgia  Thompson            18
4           5     Thomas     Moore            10


## 10. Vì sao thuật toán đúng

- **Đổi đúng tên:** `rename(columns=mapping)` thay mỗi tên cột xuất hiện làm **khóa** của `mapping` bằng giá trị tương ứng. Cả 4 tên cũ đều là khóa, nên cả 4 cột được đổi như bảng trong đề.
- **Giữ dữ liệu:** `rename` chỉ thay **nhãn** của trục cột; mảng giá trị, thứ tự hàng và thứ tự cột không bị chạm tới.
- **Không đổi nhầm cột:** việc đổi dựa vào **tên**, không phải vị trí, nên dù thứ tự cột đầu vào khác đi vẫn đúng cột nào ra đúng tên đó.
- **Cột lạ không bị hỏng:** cột không có trong `mapping` giữ nguyên tên (không báo lỗi, trừ khi ta đặt `errors="raise"`).
- **Tính chất kiểm được:** số cột, số hàng, dữ liệu từng cột (theo thứ tự) không đổi; tập tên cột mới đúng bằng 4 tên trong đề.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

- **Time:** `O(c)` để thay nhãn cột (tra từ điển `c` lần). Nếu pandas sao chép dữ liệu cho bảng mới thì thêm `O(m·c)`; với copy-on-write, dữ liệu được dùng chung cho tới khi có ai sửa.
- **Space:** `O(c)` cho danh sách/Index tên mới; `O(m·c)` nếu bảng được sao chép.
- Việc đổi tên **không phụ thuộc số hàng** khi pandas dùng chung dữ liệu; còn khi cần sao chép thì tốn tuyến tính theo kích thước bảng.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`; mỗi lần gọi truyền **bản sao** của bảng để các hàm sửa tại chỗ (Cách 3) không làm hỏng test sau.

In [20]:
impls = {
    "loop": rename_loop,
    "rename": rename_rename,
    "assign": rename_assign,
    "set_axis": rename_set_axis,
    "func": rename_func,
    "submit": renameColumns,
}

expected_example = pd.DataFrame({
    "student_id": [1, 2, 3, 4, 5],
    "first_name": ["Mason", "Ava", "Taylor", "Georgia", "Thomas"],
    "last_name": ["King", "Wright", "Hall", "Thompson", "Moore"],
    "age_in_years": [6, 7, 16, 18, 10],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(students.copy()), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
NEW_COLS = ["student_id", "first_name", "last_name", "age_in_years"]


def make(rows):
    return pd.DataFrame(rows, columns=["id", "first", "last", "age"])


def check_all(df, label):
    for key, fn in impls.items():
        out = fn(df.copy())
        assert out.columns.tolist() == NEW_COLS, (label, key, out.columns.tolist())
        assert out.shape == df.shape, (label, key)
        assert out.to_numpy().tolist() == df.to_numpy().tolist(), (label, key)
    print("PASS", label)


check_all(make([[1, "A", "B", 5]]), "một hàng")
check_all(make([[1, "A", "B", 5], [2, "C", "D", 6]]), "hai hàng")
check_all(make([[0, "", "", 0]]), "chuỗi rỗng và số 0")
check_all(make([[10**9, "Zed", "Q", 99]]), "số lớn")

PASS một hàng
PASS hai hàng
PASS chuỗi rỗng và số 0
PASS số lớn


### 12.2. Bảng rỗng, thứ tự hàng/cột, đầu vào, và phản ví dụ

In [22]:
# Bảng rỗng (chỉ có tiêu đề)
empty = make([])
for key, fn in impls.items():
    out = fn(empty.copy())
    assert out.columns.tolist() == NEW_COLS and len(out) == 0, key
print("PASS bảng rỗng: vẫn đổi được tên cột")

# Các cách "thuần" không sửa đầu vào
before = students.copy()
for key in ["loop", "rename", "set_axis", "func", "submit"]:
    impls[key](students)
    pd.testing.assert_frame_equal(students, before)
print("PASS các cách thuần không sửa đầu vào")

# Index (nhãn hàng) và thứ tự không đổi
df = students.set_index(pd.Index([10, 20, 30, 40, 50]))
out = renameColumns(df)
assert out.index.tolist() == [10, 20, 30, 40, 50]
print("PASS nhãn hàng giữ nguyên")

# Phản ví dụ 1: thiếu columns= thì không đổi gì
assert students.rename(mapping).columns.tolist() == ["id", "first", "last", "age"]
print("PASS phản ví dụ: rename(mapping) không có columns= -> không đổi cột")

# Phản ví dụ 2: gán theo vị trí khi thứ tự cột khác -> gán nhầm tên
shuffled = students[["age", "id", "first", "last"]]
wrong = shuffled.set_axis(NEW_COLS, axis=1)           # age bị gán tên student_id
right = shuffled.rename(columns=mapping)
assert wrong["student_id"].tolist() == [6, 7, 16, 18, 10]       # thực ra là tuổi
assert right["student_id"].tolist() == [1, 2, 3, 4, 5]
print("PASS phản ví dụ: đổi theo vị trí gán nhầm khi thứ tự cột khác")

PASS bảng rỗng: vẫn đổi được tên cột
PASS các cách thuần không sửa đầu vào
PASS nhãn hàng giữ nguyên
PASS phản ví dụ: rename(mapping) không có columns= -> không đổi cột
PASS phản ví dụ: đổi theo vị trí gán nhầm khi thứ tự cột khác


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh bảng ngẫu nhiên (số hàng 0–15) rồi so mọi cách với `rename` (chuẩn).

In [23]:
import random

random.seed(2885)
for trial in range(200):
    n = random.randint(0, 15)
    df = pd.DataFrame({
        "id": [random.randint(1, 10**6) for _ in range(n)],
        "first": pd.Series([random.choice(["An", "Binh", "Chi"]) for _ in range(n)], dtype="str"),
        "last": pd.Series([random.choice(["X", "Y", ""]) for _ in range(n)], dtype="str"),
        "age": [random.randint(0, 100) for _ in range(n)],
    })
    ref = df.rename(columns=mapping)
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df.copy()), ref)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `columns=` — đổi nhãn hàng thay vì cột

In [24]:
sai = students.rename({"id": "student_id"})
print("sai  :", sai.columns.tolist())                 # không đổi gì
dung = students.rename(columns={"id": "student_id"})
print("đúng :", dung.columns.tolist())

sai  : ['id', 'first', 'last', 'age']
đúng : ['student_id', 'first', 'last', 'age']


**Vì sao sai:** không có `columns=`, `rename` áp dụng cho **index** (nhãn hàng). Khóa `"id"` không khớp nhãn nào nên bị bỏ qua, không có thông báo. Hãy luôn viết `rename(columns=...)`.

### Lỗi 2: Gọi `rename` nhưng quên gán / return kết quả

In [25]:
def sai_quen_gan(students):
    students.rename(columns=mapping)          # kết quả bị vứt đi
    return students


print("sai  :", sai_quen_gan(students.copy()).columns.tolist())
print("đúng :", renameColumns(students.copy()).columns.tolist())

sai  : ['id', 'first', 'last', 'age']
đúng : ['student_id', 'first_name', 'last_name', 'age_in_years']


**Vì sao sai:** `rename` mặc định **trả bảng mới** và không sửa bảng cũ. Câu lệnh `students.rename(...)` đứng một mình tính xong rồi bỏ. Phải `return students.rename(...)` hoặc gán lại.

### Lỗi 3: `inplace=True` rồi `return` kết quả của nó

In [26]:
def sai_inplace(students):
    return students.rename(columns=mapping, inplace=True)    # trả về None!


print("sai  :", sai_inplace(students.copy()))      # None
print("đúng :", type(renameColumns(students.copy())).__name__)

sai  : None
đúng : DataFrame


**Vì sao sai:** với `inplace=True`, `rename` **sửa tại chỗ** và **trả `None`**. Bài nộp phải trả DataFrame, nên `return df.rename(..., inplace=True)` trả `None`. Cách an toàn: bỏ `inplace`, trả thẳng kết quả của `rename`.

### Lỗi 4: Gõ sai tên cột cũ — bị bỏ qua âm thầm

In [27]:
typo = students.rename(columns={"id": "student_id", "frist": "first_name", "last": "last_name", "age": "age_in_years"})
print("sai  :", typo.columns.tolist())              # 'first' vẫn là tên cũ vì gõ nhầm 'frist'

try:
    students.rename(columns={"frist": "first_name"}, errors="raise")
except KeyError as e:
    print("errors='raise' báo lỗi:", e)

sai  : ['student_id', 'first', 'last_name', 'age_in_years']
errors='raise' báo lỗi: "['frist'] not found in axis"


**Vì sao nguy hiểm:** mặc định `errors="ignore"`, khóa không tồn tại bị **lờ đi**. Khi muốn chắc chắn mọi tên cũ đều có, thêm `errors="raise"` để pandas báo `KeyError`, hoặc kiểm tra bằng `assert set(mapping) <= set(df.columns)`.

### Lỗi 5: Danh sách tên mới sai độ dài (khi gán `columns`)

In [28]:
tmp = students.copy()
try:
    tmp.columns = ["student_id", "first_name", "last_name"]       # thiếu 1 tên
except ValueError as e:
    print("ValueError:", e)

ValueError: Length mismatch: Expected axis has 4 elements, new values have 3 elements


**Vì sao sai:** gán `df.columns = list` đòi hỏi **đúng số cột**. Nếu bảng có 4 cột mà đưa 3 tên (hoặc 5 tên) sẽ báo lỗi. Với `rename` theo từ điển thì không gặp lỗi này, vì cột không có trong từ điển chỉ giữ tên cũ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đổi tên cột / rename column / đổi tiêu đề" → nghĩ tới **`df.rename(columns={...})`**.
- Công thức nhớ: **`df.rename(columns={"tên_cũ": "tên_mới"})`**, nhớ chữ `columns=`.
- Đổi **toàn bộ** tên theo thứ tự biết chắc: `df.columns = [...]` hoặc `df.set_axis([...], axis=1)`.
- Đổi hàng loạt theo quy tắc: `rename(columns=str.lower)`, `df.columns = df.columns.str.replace(" ", "_")`.
- Tránh `inplace=True`; thay vào đó gán lại hoặc `return` kết quả.
- Muốn bảo vệ khỏi gõ sai: `errors="raise"`.
- Xem tên cột bất cứ lúc nào: `df.columns.tolist()`.

## 15. Biến thể của bài

1. Chuyển **tên cột về chữ thường, thay dấu cách bằng `_`**: `df.columns = df.columns.str.lower().str.replace(" ", "_")`.
2. Thêm **tiền tố** cho mọi cột: `df.add_prefix("stu_")`; hậu tố: `df.add_suffix("_x")`.
3. Đổi tên **chỉ một cột**: `df.rename(columns={"age": "age_in_years"})`.
4. Đổi tên **nhãn hàng** (index) thay vì cột: `df.rename(index={0: "a"})`.
5. Đổi tên **cả chỉ số và cột** cùng lúc: `df.rename(index=..., columns=...)`.
6. Chọn và đổi tên cùng lúc: `df[["id", "age"]].rename(columns={"id": "student_id"})`.

Ta thử vài biến thể:

In [29]:
v = students.copy()
print("2. prefix  :", v.add_prefix("stu_").columns.tolist())
print("2. suffix  :", v.add_suffix("_x").columns.tolist())
print("3. một cột :", v.rename(columns={"age": "age_in_years"}).columns.tolist())
print("4. nhãn hàng:", v.rename(index={0: "a", 1: "b"}).index.tolist())
w = pd.DataFrame({"First Name": [1], "Last Name": [2]})
w.columns = w.columns.str.lower().str.replace(" ", "_")
print("1. chuẩn hóa:", w.columns.tolist())

2. prefix  : ['stu_id', 'stu_first', 'stu_last', 'stu_age']
2. suffix  : ['id_x', 'first_x', 'last_x', 'age_x']
3. một cột : ['id', 'first', 'last', 'age_in_years']
4. nhãn hàng: ['a', 'b', 2, 3, 4]
1. chuẩn hóa: ['first_name', 'last_name']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `dict.get` | tự tra tên mới rồi gán `columns` | `O(c)` | `O(c)` | hiểu bản chất |
| 2. `rename(columns=mapping)` | đổi theo tên | `O(c)` | `O(c)` đến `O(m·c)` | **cách nộp bài**, đổi vài cột, an toàn thứ tự |
| 3. `df.columns = [...]` | gán theo vị trí | `O(c)` | `O(c)` | đổi toàn bộ cột, biết chắc thứ tự |
| 4. `set_axis([...], axis=1)` | gán theo vị trí, trả bảng mới | `O(c)` | `O(c)` đến `O(m·c)` | method chaining, không sửa đầu vào |
| 5. `rename(columns=hàm)` | quy tắc đổi bằng hàm | `O(c)` | `O(c)` | đổi hàng loạt theo công thức |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `students.columns.tolist()`, `len(students.columns)`, `students.rename(columns={"age": "years"}).columns.tolist()`, `students.rename({"age": "years"}).columns.tolist()`, `students.add_prefix("s_").columns[0]`.

**Bài 2 (tính tay).** Với cột `["Order ID", "Unit Price"]`: viết tên cột sau khi `lower()` rồi thay dấu cách bằng `_`. Nếu `mapping = {"a": "x"}` và `df.columns = ["a", "b"]`, `rename(columns=mapping)` cho cột gì?

**Bài 3 (điền TODO).** Hoàn thành `rename_todo(df)` bằng `rename`.

In [30]:
def rename_todo(df):
    # TODO 1: tạo mapping {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    # TODO 2: return df.rename(columns=mapping)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `rename_loop2(students)` bằng vòng lặp + `dict.get` rồi gán `columns` (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `snake_case_columns(df)` trả bảng mới với mọi tên cột được `strip()`, chuyển chữ thường và thay dấu cách bằng `_`. (b) Viết `rename_checked(df, mapping)` báo `KeyError` nếu có khóa của `mapping` không nằm trong `df.columns`, ngược lại trả bảng đã đổi tên.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `['id', 'first', 'last', 'age']`, `4`, `['id', 'first', 'last', 'years']`, `['id', 'first', 'last', 'age']` (thiếu `columns=` nên không đổi), `'s_id'`. Bài 2: dùng `lower()` rồi `replace(" ", "_")`.
- **Mức 2 (vừa):** Bài 2: `['order_id', 'unit_price']`; `['x', 'b']` (cột `b` không có trong mapping nên giữ nguyên). Bài 3: hai dòng đã ghi trong comment. Bài 4: duyệt `df.columns`, `new_names.append(mapping.get(old, old))`, rồi `df.columns = new_names` trên bản sao.
- **Mức 3 (sâu):** Bài 5(a): `df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))`. Bài 5(b): `missing = set(mapping) - set(df.columns)`; nếu có thì `raise KeyError(missing)`; hoặc dùng `df.rename(columns=mapping, errors="raise")`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert students.columns.tolist() == ["id", "first", "last", "age"]
assert len(students.columns) == 4
assert students.rename(columns={"age": "years"}).columns.tolist() == ["id", "first", "last", "years"]
assert students.rename({"age": "years"}).columns.tolist() == ["id", "first", "last", "age"]
assert students.add_prefix("s_").columns[0] == "s_id"

# Bài 2
cols2 = pd.DataFrame(columns=["Order ID", "Unit Price"]).columns.str.lower().str.replace(" ", "_")
assert cols2.tolist() == ["order_id", "unit_price"]
assert pd.DataFrame(columns=["a", "b"]).rename(columns={"a": "x"}).columns.tolist() == ["x", "b"]

# Bài 3
def rename_todo_ans(df):
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    return df.rename(columns=mapping)

pd.testing.assert_frame_equal(rename_todo_ans(students), expected_example)

# Bài 4
def rename_loop2(students):
    mapping = {"id": "student_id", "first": "first_name", "last": "last_name", "age": "age_in_years"}
    df = students.copy()
    new_names = []
    for old in df.columns:
        new_names.append(mapping.get(old, old))
    df.columns = new_names
    return df

pd.testing.assert_frame_equal(rename_loop2(students), expected_example)

# Bài 5(a)
def snake_case_columns(df):
    return df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))

t = pd.DataFrame({" First Name ": [1], "AGE": [2]})
assert snake_case_columns(t).columns.tolist() == ["first_name", "age"]

# Bài 5(b)
def rename_checked(df, mapping):
    missing = set(mapping) - set(df.columns)
    if missing:
        raise KeyError(sorted(missing))
    return df.rename(columns=mapping)

assert rename_checked(students, {"id": "sid"}).columns.tolist() == ["sid", "first", "last", "age"]
try:
    rename_checked(students, {"frist": "x"})
    raise AssertionError("đáng lẽ phải báo KeyError")
except KeyError:
    pass
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi tên cột theo từ điển)

```python
import pandas as pd

def renameColumns(students: pd.DataFrame) -> pd.DataFrame:
    return students.rename(columns={
        "id": "student_id",
        "first": "first_name",
        "last": "last_name",
        "age": "age_in_years",
    })
```

**Checklist**

- [ ] Có chữ `columns=` (nếu không, pandas đổi nhãn hàng và không báo lỗi)?
- [ ] **Trả về** kết quả của `rename` (không vứt đi, không `inplace=True` rồi trả `None`)?
- [ ] Đủ 4 cặp tên và **gõ đúng chính tả** tên cũ (cân nhắc `errors="raise"` khi thử)?
- [ ] Nếu gán `df.columns = [...]`: đúng **số lượng** và đúng **thứ tự** cột?
- [ ] Dữ liệu, thứ tự hàng, thứ tự cột không đổi?
- [ ] Đã thử bảng rỗng và bảng một hàng?
- [ ] Skeleton đúng `def renameColumns(students: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?